13-09-2026

# Types of Agents
## Human in the loop Agent

In [ ]:
pip install langchain-groq langgraph langchain-community ddgs -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
from google.colab import userdata

my_api_key = userdata.get('groq_api')

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = 'openai/gpt-oss-120b',
    temperature = 0,
    api_key = my_api_key
)

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

def web_search(query: str) -> str:
    """Search the web for latest information"""
    print("🔎 web_search actually ran")
    return search.run(query)

def add(a: float, b: float) -> float:
    """Adds two numbers"""
    print("➕ add actually ran")
    return a + b

def multiply(a: float, b: float) -> float:
    """Multiplies two numbers"""
    print("✖️ multiply actually ran")
    return a * b

In [ ]:
tools = [web_search, add, multiply]
llm_with_tools = llm.bind_tools(tools)

In [ ]:
from langchain_core.messages import SystemMessage

In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

class State(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
def assistant_node(state: State):
    system = SystemMessage(content=
        "You are a helpful assistant."
        "Do not do calculations yourself, use the multiply tool or add tool"
        "Do not answer questions of factual or latest information directly, use web_search tool"
    )

    response = llm_with_tools.invoke([system] + state["messages"])

    return {
        "messages": [response]
    }

def should_continue(state: State) -> Literal["tools", END]:
    if state["messages"][-1].tool_calls:
        return "tools"
    return END

In [ ]:
memory = MemorySaver()

builder = StateGraph(State)
builder.add_node("assistant", assistant_node)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", should_continue)
builder.add_edge("tools", "assistant")

# This is the line that was missing. Halt BEFORE the tools node runs.
hitl_graph = builder.compile(
    checkpointer = memory,
    interrupt_before = ['tools']
)

In [ ]:
thread_id = 'kartik'

query = 'What is price of silver in India?'
config = {"configurable": {"thread_id": thread_id}}
hitl_graph.invoke({"messages": [("user", query)]}, config=config)['messages'][-1].content

''

In [ ]:
hitl_graph.invoke(None, config=config)['messages'][-1].content

🔎 web_search actually ran


'The current market price of silver in India (99.9\u202f% purity) is about **₹245\u202fper gram**【web_search】.'

In [ ]:
# Human in the loop agent stops before using any tools
# But if you call it again, it is taken as a approval from your side
# When calling second time, DO NOT send any message with it

In [ ]:
from langchain_core.messages import ToolMessage

def run_with_approval(query: str, thread_id: str):
    config = {"configurable": {"thread_id": thread_id}}

    # First run. It will stop right before the first tool call.
    hitl_graph.invoke({"messages": [("user", query)]}, config=config)

    # Keep going until the graph has nothing left to do
    while True:
        snapshot = hitl_graph.get_state(config)

        # snapshot.next is empty when the graph has finished
        if not snapshot.next:
            break

        last_message = snapshot.values["messages"][-1]

        print("\n⏸️  PAUSED. The agent wants to call:")
        for call in last_message.tool_calls:
            print(f"   Tool : {call['name']}")
            print(f"   Args : {call['args']}")

        decision = input("Approve? (y/n): ").strip().lower()

        if decision == "y":
            # Passing None means: resume from where you stopped
            hitl_graph.invoke(None, config=config)
        else:
            # Tell the agent it was denied, then let it respond
            denials = [
                ToolMessage(
                    content = "Tool call denied by the human. Answer without it.",
                    tool_call_id = call["id"]
                )
                for call in last_message.tool_calls
            ]
            hitl_graph.update_state(config, {"messages": denials}, as_node="tools")
            hitl_graph.invoke(None, config=config)

    final = hitl_graph.get_state(config).values
    print("\n🤖 Final Answer:")
    print(final["messages"][-1].content)

In [ ]:
run_with_approval("What is 99 * 88?", thread_id="hitl_1")


⏸️  PAUSED. The agent wants to call:
   Tool : multiply
   Args : {'a': 99, 'b': 88}
Approve? (y/n): y
✖️ multiply actually ran

🤖 Final Answer:
99 × 88 = 8712.


In [ ]:
# Test 2: two tool calls. Approve the first, and it continues
# to the next pause automatically without you re-running anything.
run_with_approval(
    "Search the current price of silver per gram in India, then multiply it by 40.",
    thread_id="hitl_2"
)


⏸️  PAUSED. The agent wants to call:
   Tool : web_search
   Args : {'query': 'current price of silver per gram in India'}
Approve? (y/n): y
🔎 web_search actually ran

⏸️  PAUSED. The agent wants to call:
   Tool : multiply
   Args : {'a': 245, 'b': 40}
Approve? (y/n): y
✖️ multiply actually ran

🤖 Final Answer:
The latest information I found shows that the price of silver in India is **₹245 per gram**.  

Multiplying this price by 40 grams:

**₹245 × 40 = ₹9,800**

So, 40 grams of silver would cost approximately **₹9,800**.


## ReAct Agent

Customer Support Bot
Delivery support bot

#### Scenario: A delivery support bot. It does not know the refund amount until it first checks how late the order is. That dependency is exactly why ReAct exists: step 2 cannot be decided before step 1 returns.

In [ ]:
pip install langchain-groq langgraph -q

In [ ]:
from google.colab import userdata
from langchain_groq import ChatGroq

my_api_key = userdata.get('groq_api')

llm = ChatGroq(
    model = 'openai/gpt-oss-20b',
    temperature = 0,
    api_key = my_api_key
)

In [ ]:
# Fake company database. In real life this is an API call.
ORDERS = {
    "NST1042": {"item": "Laptop Stand", "price": 2400, "days_late": 6},
    "NST1077": {"item": "Wireless Mouse", "price": 900, "days_late": 0},
}

def track_order(order_id: str) -> str:
    """Look up an order and report how many days late it is"""
    print("📦 track_order called:", order_id)
    order = ORDERS.get(order_id)
    if not order:
        return "Order not found"
    return f"{order['item']}, price {order['price']}, {order['days_late']} days late"

def calculate_refund(price: float, days_late: int) -> float:
    """Calculate refund: 5 percent of price per day late, capped at 30 percent"""
    print("💰 calculate_refund called:", price, days_late)
    percent = min(days_late * 5, 30)
    return price * percent / 100

In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

tools = [track_order, calculate_refund]
llm_with_tools = llm.bind_tools(tools)

class State(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
# The ReAct prompt. The reasoning instruction is doing real work here.
react_prompt = (
    "You are a delivery support agent. Solve the request step by step. "
    "Before each action, briefly state what information you still need. "
    "Use the tools to get facts, never guess numbers. Stop once you can answer."
)

def reason_node(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

def react_route(state: State) -> Literal["tools", END]:
    if state["messages"][-1].tool_calls:
        return "tools"
    return END

In [ ]:
builder = StateGraph(State)
builder.add_node("reason", reason_node)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "reason")
builder.add_conditional_edges("reason", react_route)
builder.add_edge("tools", "reason")

react_agent = builder.compile()

In [ ]:
query = "Order NST1042 is late. Check it and tell me my refund amount."

result = react_agent.invoke({
    "messages": [("system", react_prompt), ("user", query)]
})

print(result["messages"][-1].content)

📦 track_order called: NST1042
Your order **NST1042** is 6 days late.  
The refund policy is 5 % of the order price per day late, capped at 30 %.  

- 5 % × 6 days = 30 % of the price  
- 30 % of ₹2,400 = **₹720**

**Refund amount: ₹720**.


## Plan and Execute
#### Scenario: Writing a launch brief has known stages. Nothing needs to be discovered mid-task, so planning everything upfront gives cleaner structure than ReAct's step-by-step drift.

In [ ]:
pip install langchain-groq langgraph -q

In [ ]:
from google.colab import userdata
from langchain_groq import ChatGroq

my_api_key = userdata.get('groq_api')

llm = ChatGroq(
    model = 'openai/gpt-oss-20b',
    temperature = 0,
    api_key = my_api_key
)

In [ ]:
planner_prompt = (
    "You are a planner. Break the task into 4 concrete steps. "
    "Reply ONLY as a numbered list, one step per line, like '1. ...'. "
    "No introduction, no explanation."
)

def parse_plan(text: str) -> list:
    steps = []
    for line in text.splitlines():
        line = line.strip()
        if line and line[0].isdigit():
            steps.append(line.split(".", 1)[1].strip())
    return steps

In [ ]:
class PlanState(TypedDict):
    task: str
    plan: list
    step_index: int
    results: list

In [ ]:
def planner_node(state: PlanState):
    text = llm.invoke(planner_prompt + "\n\nTask: " + state["task"]).content
    plan = parse_plan(text)

    print("🗂️ PLAN:")
    for i, s in enumerate(plan, 1):
        print(f"   {i}. {s}")

    return {"plan": plan, "step_index": 0, "results": []}

def executor_node(state: PlanState):
    i = state["step_index"]
    step = state["plan"][i]

    # Pass earlier results forward so steps build on each other
    previous = "\n".join(state["results"])
    context = f"\n\nWork completed so far:\n{previous}" if previous else ""

    print(f"⚙️ executing step {i + 1}: {step}")

    out = llm.invoke(
        f"Overall task: {state['task']}{context}\n\n"
        f"Carry out ONLY this step and report the result briefly:\n{step}"
    ).content

    return {"results": state["results"] + [out], "step_index": i + 1}

def more_steps(state: PlanState):
    if state["step_index"] < len(state["plan"]):
        return "executor"
    return END

In [ ]:
builder = StateGraph(PlanState)
builder.add_node("planner", planner_node)
builder.add_node("executor", executor_node)

builder.add_edge(START, "planner")
builder.add_edge("planner", "executor")
builder.add_conditional_edges("executor", more_steps)

plan_execute_agent = builder.compile()

In [ ]:
task = "Create a launch brief for a new 8-week Agentic AI course for working data scientists."

result = plan_execute_agent.invoke({
    "task": task,
    "plan": [],
    "step_index": 0,
    "results": []
})

for i, r in enumerate(result["results"], 1):
    print(f"\n===== STEP {i} OUTPUT =====")
    print(r)

🗂️ PLAN:
   1. Identify the target audience (working data scientists) and define clear learning objectives for the 8‑week Agentic AI course.
   2. Draft a detailed curriculum outline, including weekly modules, key topics, hands‑on projects, and assessment methods.
   3. Create a launch marketing plan: craft messaging, select channels (email, LinkedIn, webinars), and schedule promotional activities.
   4. Assemble launch logistics: finalize course materials, set up enrollment platform, coordinate instructors, and plan a kickoff event.
⚙️ executing step 1: Identify the target audience (working data scientists) and define clear learning objectives for the 8‑week Agentic AI course.
⚙️ executing step 2: Draft a detailed curriculum outline, including weekly modules, key topics, hands‑on projects, and assessment methods.
⚙️ executing step 3: Create a launch marketing plan: craft messaging, select channels (email, LinkedIn, webinars), and schedule promotional activities.
⚙️ executing step 4: A

# Streamlit: Python's way of creating websites

In [ ]:
!pip install streamlit -q

In [ ]:
%%writefile app.py
import streamlit as st

# 1. Titles
st.title('🤖 My First AI Web App')
st.header('Welcome to the Dashboard')
st.subheader('Sub Section: Data Analysis')

# 2. Standard Text
st.write("This is a standard text. st.write() is the 'print' function of Streamlit")
st.text("This is a fixed-width text, good for showing raw code or logs") # justified alignment

# 3. Formatted Markdown (Crucial for AI Outputs)
st.markdown("**Bold Text**, *Italic Text*, and even [Links](https://streamlit.io).")

# 4. Status Messages (Success, Error, warning, info)
st.success('The AI has successfully processed your request')
st.error('Critical Error: API Key Missing')
st.warning('Warning: The model might hallucinate')
st.info('Fun Fact: LLama-8 has 8 billion parameters')

# Button
st.title('Button Test')

st.write('Click the button below to trigger an action')

# Button Logic
if st.button('Generate Random Alert:'):
    st.success('You clicked the button!')
else:
    st.info('Waiting for you to click')

Overwriting app.py


In [ ]:
!pip install streamlit pyngrok -q

In [ ]:
from google.colab import userdata
from pyngrok import ngrok

ngrok.set_auth_token(userdata.get("NGROK_AUTH_TOKEN"))

In [ ]:
import subprocess
import time

# Start Streamlit in the background
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8502", "--server.headless", "true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Give it a couple seconds to boot
time.sleep(3)

# Open ngrok tunnel to port 8501
public_url = ngrok.connect(8502)
print(f"✅ Your app is live at: {public_url}")

✅ Your app is live at: NgrokTunnel: "https://cleruchial-causally-toshiko.ngrok-free.dev" -> "http://localhost:8502"


In [ ]:
from pyngrok import ngrok

ngrok.kill()

In [ ]:
# Practice Challenge 1: "The AI Status Dashboard"
# Scenario: You are building the control panel for the Autonomous Market Researcher.
# You need a simple UI that shows the status of the system and has a "Run Diagnostics" button.

# Your Task:

# Create a new app.py script.

# Give it a title: "System Operations Center".

# Use st.markdown to write a short bulleted list of the tools currently online (e.g., Web Search, Calculator, Scraper).

# Add a button that says "Run System Diagnostics".

# Logic: * If the button is clicked, show an st.success message saying "All systems nominal. API connection verified."

# If it is not clicked, show an st.warning message saying "Diagnostics pending. Please click the button."

In [ ]:
%%writefile app.py

import streamlit as st

# 1. Title
st.title('🖥️ System Operations Center')

# 2. Tools Currently Online
st.markdown("""
### 🟢 Tools Currently Online

- 🌐 Web Search
- 🧮 Calculator
- 🔎 Scraper
""")

# 3. Button
st.title('System Diagnostics')

st.write('Click the button below to check the system status')

# 4. Button Logic
if st.button('Run System Diagnostics'):
    st.success('All systems nominal. API connection verified.')
else:
    st.warning('Diagnostics pending. Please click the button.')

Overwriting app.py


In [ ]:
!pip install streamlit pyngrok -q

In [ ]:
from google.colab import userdata
from pyngrok import ngrok

ngrok.set_auth_token(userdata.get("NGROK_AUTH_TOKEN"))

In [ ]:
import subprocess
import time

# Start Streamlit in the background
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8502", "--server.headless", "true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Give it a couple seconds to boot
time.sleep(3)

# Open ngrok tunnel to port 8502
public_url = ngrok.connect(8502)

print(f"✅ Your app is live at: {public_url}")

✅ Your app is live at: NgrokTunnel: "https://cleruchial-causally-toshiko.ngrok-free.dev" -> "http://localhost:8502"


In [ ]:
from pyngrok import ngrok

ngrok.kill()

In [ ]:
# Adding Extra component --- this code is not a challenge 1 code----

%%writefile app.py
import streamlit as st

# --- PAGE CONFIGURATION ---
# This must be the first Streamlit command. It sets the browser tab title and width.
st.set_page_config(page_title="Agent Builder", layout="centered") # layout="centered" it means our website is on center

st.title("⚙️ The Agent Configuration Panel")
st.write("Design your AI persona before launching the chat.")

# The Sidebar (Layout) # side bar like chatGpt have left side side bar
with st.sidebar:
    st.header('Security and Settings')

    # Password input hides the text
    api_key = st.text_input('Groq API Key', type='password')  # password show like *****

    # Slider for numbers (min_value, max, default value)
    temperature = st.slider('Creativity (Temperature):', min_value=0.0, max_value=1.0, value=0.7)

    st.info("The sidebar is great for settings that shouldn't clutter the main screen.")

# Columns
col1, col2 = st.columns(2)

with col1:
    st.subheader('Agent Identity')
    agent_name = st.text_input('Name your agent:', value='DataBot')

    # DropDown
    agent_role = st.selectbox("Select Agent Role:", ['Data Analyst', 'Copywriter', 'Python Tutor'])

with col2:
    st.subheader("Agent Behavior")
    # Text area for long paragraphs
    system_instructions = st.text_area("Custom System Instructions:", height=110,
                                       placeholder="E.g., Always answer in bullet points...")


# --- 3. THE OUTPUT (Testing the inputs) ---
st.write("---")
st.subheader("Preview Your Configuration")

# We only show the success box if they actually typed a name
if agent_name:
    st.success(f"**{agent_name}** is ready to be deployed as a **{agent_role}**.")

    # We display exactly how the System Prompt will look to the LLM
    st.code(f"""
    System Prompt:
    You are {agent_name}, an expert {agent_role}.
    Your creativity level is set to {temperature}.
    Additional Instructions: {system_instructions}
    """, language="text")

Overwriting app.py


In [ ]:
!pip install streamlit pyngrok -q

In [ ]:
from google.colab import userdata
from pyngrok import ngrok

ngrok.set_auth_token(userdata.get("NGROK_AUTH_TOKEN"))

In [ ]:
import subprocess
import time

# Start Streamlit in the background
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8502", "--server.headless", "true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Give it a couple seconds to boot
time.sleep(3)

# Open ngrok tunnel to port 8501
public_url = ngrok.connect(8502)
print(f"✅ Your app is live at: {public_url}")

✅ Your app is live at: NgrokTunnel: "https://cleruchial-causally-toshiko.ngrok-free.dev" -> "http://localhost:8502"


In [ ]:
from pyngrok import ngrok

ngrok.kill()

In [ ]:
# Practice Challenge 2: "The Calculator UI"
# Scenario: Before we hook up our complex LangGraph Math Agent to a UI, we need to build the "shell" for it.
# You need to build a simple UI for a calculator tool.

# Your Task:

# Create a new %%writefile app.py script.

# Give it a title: "🧮 Agentic Calculator".

# Create two columns.

# In Column 1, use st.number_input to ask for the "First Number".

# In Column 2, use st.number_input to ask for the "Second Number".

# Below the columns, use a dropdown (st.selectbox) to ask for the Operation: "Add", "Subtract", "Multiply", or "Divide".

# Add a button that says "Calculate".

# The Logic: If the button is clicked, do the math using standard Python logic (no LLM needed yet) and display the result using st.success().

# Hint: st.number_input("Label", value=0) forces the user to type a number, preventing text errors!

In [ ]:
%%writefile app.py

import streamlit as st

# 1. Title
st.title('🧮 Agentic Calculator')

# 2. Two Columns
col1, col2 = st.columns(2)

with col1:
    first_number = st.number_input('First Number', value=0)

with col2:
    second_number = st.number_input('Second Number', value=0)

# 3. Operation
operation = st.selectbox(
    'Operation',
    ['Add', 'Subtract', 'Multiply', 'Divide']
)

# 4. Button
if st.button('Calculate'):

    # 5. Calculation Logic
    if operation == 'Add':
        result = first_number + second_number

    elif operation == 'Subtract':
        result = first_number - second_number

    elif operation == 'Multiply':
        result = first_number * second_number

    elif operation == 'Divide':
        if second_number != 0:
            result = first_number / second_number
        else:
            st.error('Cannot divide by zero.')
            result = None

    # 6. Display Result
    if result is not None:
        st.success(f'Result: {result}')

Overwriting app.py


In [ ]:
!pip install streamlit pyngrok -q

In [ ]:
from google.colab import userdata
from pyngrok import ngrok

ngrok.set_auth_token(userdata.get("NGROK_AUTH_TOKEN"))

In [ ]:
import subprocess
import time

# Start Streamlit in the background
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8502", "--server.headless", "true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Give it a couple seconds to boot
time.sleep(3)

# Open ngrok tunnel to port 8501
public_url = ngrok.connect(8502)
print(f"✅ Your app is live at: {public_url}")

✅ Your app is live at: NgrokTunnel: "https://cleruchial-causally-toshiko.ngrok-free.dev" -> "http://localhost:8502"


In [ ]:
from pyngrok import ngrok

ngrok.kill()